In [1]:
try:
    import google.colab  # noqa: F401

    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import numpy as np
import torch
from IPython.display import display  # noqa: A004
from maite_datasets.object_detection import VOCDetection
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms.v2 import GaussianNoise

from dataeval import Embeddings, Metadata
from dataeval.core import label_parity
from dataeval.extractors import TorchExtractor
from dataeval.shift import DriftChunkedOutput, DriftMMD, DriftMVDC, DriftUnivariate

# Set a random seed
rng = np.random.default_rng(213)

# Set default torch device for notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)

In [3]:
resnet = resnet18(weights=ResNet18_Weights.DEFAULT, progress=False)

# Replace the final fully connected layer with a Linear layer
resnet.fc = torch.nn.Linear(resnet.fc.in_features, 128)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


In [4]:
# Load the training dataset
train_ds = VOCDetection("./data", year="2012", image_set="train", download=True)
print(train_ds)
print(f"Image 0 shape: {train_ds[0][0].shape}")

VOCDetection Dataset
--------------------
    Year: 2012
    Transforms: []
    Image Set: train
    Metadata: {'id': 'VOCDetection_train', 'index2label': {0: 'aeroplane', 1: 'bicycle', 2: 'bird', 3: 'boat', 4: 'bottle', 5: 'bus', 6: 'car', 7: 'cat', 8: 'chair', 9: 'cow', 10: 'diningtable', 11: 'dog', 12: 'horse', 13: 'motorbike', 14: 'person', 15: 'pottedplant', 16: 'sheep', 17: 'sofa', 18: 'train', 19: 'tvmonitor'}, 'split': 'train'}
    Path: /builds/jatic/aria/dataeval/docs/source/notebooks/data/vocdataset/VOCdevkit/VOC2012
    Size: 5717
Image 0 shape: (3, 442, 500)


In [5]:
# Load the "operational" dataset
operational_ds = VOCDetection("./data", year="2012", image_set="val", download=True)
print(operational_ds)
print(f"Image 0 shape: {train_ds[0][0].shape}")

VOCDetection Dataset
--------------------
    Year: 2012
    Transforms: []
    Image Set: val
    Metadata: {'id': 'VOCDetection_val', 'index2label': {0: 'aeroplane', 1: 'bicycle', 2: 'bird', 3: 'boat', 4: 'bottle', 5: 'bus', 6: 'car', 7: 'cat', 8: 'chair', 9: 'cow', 10: 'diningtable', 11: 'dog', 12: 'horse', 13: 'motorbike', 14: 'person', 15: 'pottedplant', 16: 'sheep', 17: 'sofa', 18: 'train', 19: 'tvmonitor'}, 'split': 'val'}
    Path: /builds/jatic/aria/dataeval/docs/source/notebooks/data/vocdataset/VOCdevkit/VOC2012
    Size: 5823
Image 0 shape: (3, 442, 500)


In [6]:
# Define pretrained model transformations
transforms = ResNet18_Weights.DEFAULT.transforms()

# Create extractor with model and transforms
extractor = TorchExtractor(resnet, transforms=transforms)

# Create training batches and targets
train_embs = Embeddings(train_ds, extractor=extractor, batch_size=64)

# Create operational batches and targets
operational_embs = Embeddings(operational_ds, extractor=extractor, batch_size=64)

In [7]:
print(f"({len(train_embs)}, {train_embs[0].shape})")  # (5717, shape)
print(f"({len(operational_embs)}, {operational_embs[0].shape})")  # (5823, shape)

(5717, (128,))
(5823, (128,))


In [8]:
# A type alias for all of the drift detectors
DriftDetector = DriftMMD | DriftUnivariate | DriftMVDC

# Create a mapping for the detectors to iterate over
detectors: dict[str, DriftDetector] = {
    "MMD": DriftMMD().fit(train_embs),
    "CVM": DriftUnivariate(method="cvm").fit(train_embs),
    "KS": DriftUnivariate(method="ks").fit(train_embs),
    "MVDC": DriftMVDC().fit(train_embs),
}

In [9]:
# Iterate and print the name of the detector class and its boolean drift prediction
for name, detector in detectors.items():
    print(f"{name} detected drift? {detector.predict(operational_embs).drifted}")

MMD detected drift? False


CVM detected drift? False


KS detected drift? False


MVDC detected drift? False


In [10]:
# Define transform with added gaussian noise
noisy_transforms = [transforms, GaussianNoise()]

# Create extractor with noisy transforms
noisy_extractor = TorchExtractor(resnet, transforms=noisy_transforms)

# Applies gaussian noise to images before processing
noisy_embs = Embeddings(operational_ds, extractor=noisy_extractor, batch_size=64)

In [11]:
# Iterate and print the name of the detector class and its boolean drift prediction
for name, detector in detectors.items():
    print(f"{name} detected drift? {detector.predict(noisy_embs).drifted}")

MMD detected drift? True


CVM detected drift? True


KS detected drift? True


MVDC detected drift? True


In [12]:
# Build a combined array: first 40% clean, last 60% noisy
n_operational = len(operational_embs)
split_idx = int(n_operational * 0.4)

combined_embs = np.concatenate([operational_embs[:split_idx], noisy_embs[split_idx:]])
print(f"Combined shape: {combined_embs.shape} (clean: {split_idx}, noisy: {n_operational - split_idx})")

Combined shape: (5823, 128) (clean: 2329, noisy: 3494)


In [13]:
# Re-fit detectors with chunking enabled (5 chunks each)
chunked_detectors: dict[str, DriftDetector] = {
    "MMD": DriftMMD().fit(train_embs, chunk_count=5),
    "CVM": DriftUnivariate(method="cvm").fit(train_embs, chunk_count=5),
    "KS": DriftUnivariate(method="ks").fit(train_embs, chunk_count=5),
    "MVDC": DriftMVDC(threshold=(0.45, 0.65)).fit(train_embs, chunk_count=5),
}

In [14]:
for name, detector in chunked_detectors.items():
    result = detector.predict(combined_embs)
    print(f"\n{name} - Overall drift detected: {result.drifted} (metric: {result.metric_name})")
    if isinstance(result, DriftChunkedOutput):
        display(result.chunk_results)


MMD - Overall drift detected: True (metric: mmd2)


key,index,start_index,end_index,value,upper_threshold,lower_threshold,drifted
str,i64,i64,i64,f64,f64,f64,bool
"""[0:1143]""",0,0,1143,0.006338,0.009111,-0.004368,false
"""[1144:2287]""",1,1144,2287,0.00029,0.009111,-0.004368,false
"""[2288:3431]""",2,2288,3431,0.097267,0.009111,-0.004368,true
"""[3432:4575]""",3,3432,4575,0.103885,0.009111,-0.004368,true
"""[4576:5822]""",4,4576,5822,0.104817,0.009111,-0.004368,true



CVM - Overall drift detected: True (metric: cvm_distance)


key,index,start_index,end_index,value,upper_threshold,lower_threshold,drifted
str,i64,i64,i64,f64,f64,f64,bool
"""[0:1143]""",0,0,1143,1.509423,2.017732,0.0,false
"""[1144:2287]""",1,1144,2287,0.223395,2.017732,0.0,false
"""[2288:3431]""",2,2288,3431,20.883604,2.017732,0.0,true
"""[3432:4575]""",3,3432,4575,22.319241,2.017732,0.0,true
"""[4576:5822]""",4,4576,5822,24.329414,2.017732,0.0,true



KS - Overall drift detected: True (metric: ks_distance)


key,index,start_index,end_index,value,upper_threshold,lower_threshold,drifted
str,i64,i64,i64,f64,f64,f64,bool
"""[0:1143]""",0,0,1143,0.058145,0.07352,0.011325,false
"""[1144:2287]""",1,1144,2287,0.029978,0.07352,0.011325,false
"""[2288:3431]""",2,2288,3431,0.181546,0.07352,0.011325,true
"""[3432:4575]""",3,3432,4575,0.187235,0.07352,0.011325,true
"""[4576:5822]""",4,4576,5822,0.190271,0.07352,0.011325,true



MVDC - Overall drift detected: True (metric: auroc)


key,index,start_index,end_index,value,upper_threshold,lower_threshold,drifted
str,i64,i64,i64,f64,f64,f64,bool
"""[0:1143]""",0,0,1143,0.613517,0.65,0.45,false
"""[1144:2287]""",1,1144,2287,0.507449,0.65,0.45,false
"""[2288:3431]""",2,2288,3431,0.972012,0.65,0.45,true
"""[3432:4575]""",3,3432,4575,0.995999,0.65,0.45,true
"""[4576:5822]""",4,4576,5822,0.996487,0.65,0.45,true


In [15]:
# Get the metadata for each dataset
train_md = Metadata(train_ds)
operational_md = Metadata(operational_ds)

# The VOC dataset has 20 classes
label_parity(train_md.class_labels, operational_md.class_labels, num_classes=20)["p_value"]

0.949856067521638